In [0]:
# =============================================================
# nb_silver_to_gold_sales.py
# Layer     : Gold
# Audience  : Sales Executive Team
# Purpose   : Orchestrates the Sales monthly summary Gold table.
#
# Output table : gold/gold_sales_summary
# Grain        : sale_month × sales_channel × device_category
#                × country × state
#
# Sources (Silver):
#   sales    → sale_price, discount_amount, sales_channel
#   owners   → country, state (geographic breakdown)
#   devices  → product_id (to get category)
#   products → device_category, product_name
#
# ADF params: storage_account
# =============================================================

# ── CELL 1: Widgets ───────────────────────────────────────────
dbutils.widgets.removeAll()
dbutils.widgets.text("storage_account", "petiot")


def abfss(container: str, path: str = "", storage_account: str | None = None) -> str:
    active_storage_account = storage_account or dbutils.widgets.get("storage_account")
    clean_path = path.lstrip("/")
    base = f"abfss://{container}@{active_storage_account}.dfs.core.windows.net"
    return f"{base}/{clean_path}" if clean_path else f"{base}/"

print("=" * 60)
print("  nb_silver_to_gold_sales")
print("  Output: gold_sales_summary")
print("  Grain : sale_month × channel × category × geography")
print("=" * 60)

# ── CELL 2: Auth ──────────────────────────────────────────────
%run /Workspace/Shared/pet-analytics/shared/set_auth_context.py.ipynb

# ── CELL 3: Load specialists ──────────────────────────────────
%run /Workspace/Shared/pet-analytics/gold/gold_reader.ipynb
%run /Workspace/Shared/pet-analytics/gold/aggregator.ipynb
%run /Workspace/Shared/pet-analytics/gold/gold_writer.ipynb

# ── CELL 4: Read Silver tables ────────────────────────────────
from pyspark.sql.functions import (
    col, date_trunc, round as spark_round
)

print("\n[sales] Reading Silver tables...")
sales_df    = read_sales()
owners_df   = read_owners()
devices_df  = read_devices()
products_df = read_products()

print(f"  sales    : {sales_df.count():,}")
print(f"  owners   : {owners_df.count():,}")
print(f"  devices  : {devices_df.count():,}")
print(f"  products : {products_df.count():,}")

# ── CELL 5: Stage 1 — Enrich sales with dimension labels ──────
# Sales is small (10,000 rows) — safe to join dimensions directly
# without pre-aggregating first. The join does not fan out because
# sales.device_id is unique (one sale per device).
print("\n[sales] Stage 1: Enriching sales with dimension labels...")

# Slim device and product joins — only the columns we need
devices_slim  = devices_df.select("device_id", "product_id")
products_slim = products_df.select("product_id", "device_category", "product_name", "msrp")
owners_slim   = owners_df.select("owner_id", "country", "state", "city")

sales_enriched = (
    sales_df
    # Join devices to get product_id
    .join(devices_slim,  "device_id",  "left")
    # Join products to get device_category
    .join(products_slim, "product_id", "left")
    # Join owners for geography
    # Note: sales already has city/state from generation but
    # we use owners for country which is more reliable
    .join(owners_slim.drop("city","state"), "owner_id", "left")
    # Derive sale_month for monthly grain
    .withColumn("sale_month",
        date_trunc("month", col("sale_date")).cast("date"))
    # Derive net_revenue
    .withColumn("net_revenue",
        spark_round(col("sale_price") - col("discount_amount"), 2))
)

print(f"  Enriched sales rows: {sales_enriched.count():,}")

# ── CELL 6: Stage 2 — Aggregate to Gold grain ─────────────────
# Now aggregate the enriched sales to the Gold grain:
# sale_month × sales_channel × device_category × country × state
print("\n[sales] Stage 2: Aggregating to monthly summary...")

gold_df = agg_sales_summary(sales_enriched)
print(f"  Gold summary rows: {gold_df.count():,}")

# ── CELL 7: Apply null defaults ───────────────────────────────
gold_df = apply_null_defaults(gold_df, "gold_sales_summary")

# ── CELL 8: Add audit + write ─────────────────────────────────
gold_df     = add_gold_audit(gold_df)
final_count = write_gold(gold_df, "gold_sales_summary")

# ── CELL 9: Preview ───────────────────────────────────────────
print("\n[sales] Sample output:")
spark.read.format("delta") \
    .load(abfss("gold", "gold_sales_summary")) \
    .select("sale_month", "sales_channel", "device_category",
            "country", "total_units_sold",
            "gross_revenue", "net_revenue", "discount_rate_pct") \
    .orderBy("sale_month", "gross_revenue", ascending=[True, False]) \
    .show(5, truncate=False)

print(f"\n{'='*60}")
print(f"  Gold complete: gold_sales_summary")
print(f"  Rows: {final_count:,}")
print(f"{'='*60}")

dbutils.notebook.exit(f"SUCCESS|gold_sales_summary|{final_count}")